<a href="https://colab.research.google.com/github/emiryucedag/Systematic-Investment-Strategy-with-Markowitz-Allocator/blob/main/471_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd

# 1. Veriyi okuma ve 'Date' sütununu zaman indeksine (datetime) çevirme
df = pd.read_csv("YAP471_Temiz_Kapanis_Verileri.csv")
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)

# 2. Z-Score Hesaplaması
# 20 günlük kayan (rolling) pencere üzerinden ortalama ve standart sapma
ma_20 = df.rolling(window=20).mean()
std_20 = df.rolling(window=20).std()

# Formülü uygulayarak Z-Score matrisini elde etme
z_scores = (df - ma_20) / std_20

# İlk 25 güne ve son günlere bakalım
print(z_scores.head(25))

                AAPL      MSFT      AMZN     GOOGL      TSLA      NVDA
Date                                                                  
2021-03-03       NaN       NaN       NaN       NaN       NaN       NaN
2021-03-04       NaN       NaN       NaN       NaN       NaN       NaN
2021-03-05       NaN       NaN       NaN       NaN       NaN       NaN
2021-03-08       NaN       NaN       NaN       NaN       NaN       NaN
2021-03-09       NaN       NaN       NaN       NaN       NaN       NaN
2021-03-10       NaN       NaN       NaN       NaN       NaN       NaN
2021-03-11       NaN       NaN       NaN       NaN       NaN       NaN
2021-03-12       NaN       NaN       NaN       NaN       NaN       NaN
2021-03-15       NaN       NaN       NaN       NaN       NaN       NaN
2021-03-16       NaN       NaN       NaN       NaN       NaN       NaN
2021-03-17       NaN       NaN       NaN       NaN       NaN       NaN
2021-03-18       NaN       NaN       NaN       NaN       NaN       NaN
2021-0

In [4]:
# Z-skorlarını -1 ile çarparak Beklenen Getiri Proxy'sine çeviriyoruz
expected_returns_proxy = z_scores * -1

# Sorunsuz kullanması için ilk 19 gündeki NaN (boş) verileri atıyoruz
expected_returns_proxy.dropna(inplace=True)

# Çıktıyı yeni ve profesyonel bir isimle kaydediyoruz
expected_returns_proxy.to_csv("YAP471_ZScore_Expected_Returns.csv")

print("Gidecek verinin ilk 5 günü:")
print(expected_returns_proxy.head())

Gidecek verinin ilk 5 günü:
                AAPL      MSFT      AMZN     GOOGL      TSLA      NVDA
Date                                                                  
2021-03-30  0.733271  0.440287  0.130543 -0.056507  0.427620 -0.238385
2021-03-31 -0.368465 -0.604251 -0.575656 -0.584672 -0.410937 -1.296985
2021-04-01 -0.712430 -2.321058 -1.780617 -2.477413 -0.197300 -2.024444
2021-04-05 -1.834357 -2.975025 -2.520373 -3.351775 -0.902504 -2.016887
2021-04-06 -1.901553 -2.297084 -2.283794 -2.469782 -0.934339 -1.799743


Projemizde sinyal üretimi aşamasında, ortalamaya dönüş (mean reversion) stratejisine temel oluşturması amacıyla 20 günlük kayan Z-skoru (rolling Z-score) kullanılmıştır. Sinyal hesaplaması $z_t = (P_t - MA_{20}) / \sigma_{20}$ formülüyle gerçekleştirilmiştir. Modelin doğası gereği, ilk 19 işlem gününde geriye dönük 20 günlük veri seti tam olarak oluşmadığı için hesaplama yapılamamış ve bu günler veri setinde boş (NaN) bırakılarak analiz dışı tutulmuştur. Bu yaklaşım, sinyal inşası sırasında geleceği görme hatasından (look-ahead bias) kesinlikle kaçınılmasını sağlamış ve geriye dönük testin temiz bir kayan örneklem dışı (rolling out-of-sample) tasarımla ilerlemesini güvence altına almıştır.

Stratejimizde Z-skoru hesaplanırken 20 günlük kayan pencere (rolling window) kullanılmıştır. 'Look-ahead bias' (geleceği görme hatası) oluşmaması için ilk 19 günlük veri setten çıkarılmıştır. Markowitz optimizer'ının düşük Z-skorlu (ucuz) hisselere ağırlık verebilmesi için, Z-skoru değerleri -1 ile çarpılarak 'Beklenen Getiri' (Expected Return Proxy) formatına dönüştürülmüştür.